# 01c — Leakage Audit
Validate features against the whitelist using `src.data.leakage_audit`.

In [ ]:
import os
from pathlib import Path

# Find project root (contains pyproject.toml)
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
import pandas as pd

from src.utils.config import load_main_config, load_feature_whitelist, resolve_path
from src.data.loader import load_raw_data
from src.data.leakage_audit import (
    get_problem_a_features, get_problem_b_features, validate_no_leakage,
)

config = load_main_config()
whitelist = load_feature_whitelist()
df = load_raw_data(config)
print(f'Loaded {len(df)} records with {len(df.columns)} columns.')

## 1. Whitelist Verification

In [ ]:
prob_a = get_problem_a_features(df, whitelist)
prob_b = get_problem_b_features(df, whitelist)

print(f'Problem A: {len(prob_a)} approved features')
print(f'Problem B: {len(prob_b)} approved features')

print('\nProblem A features:')
for f in sorted(prob_a):
    print(f'  {f}')

## 2. Automated Leakage Check

In [ ]:
is_clean_a, leaks_a = validate_no_leakage(prob_a, problem='A', whitelist=whitelist)
is_clean_b, leaks_b = validate_no_leakage(prob_b, problem='B', whitelist=whitelist)

if is_clean_a:
    print('✅ Problem A: No leakage detected.')
else:
    for l in leaks_a:
        print(f'  ❌ LEAK: {l}')

if is_clean_b:
    print('✅ Problem B: No leakage detected.')
else:
    for l in leaks_b:
        print(f'  ❌ LEAK: {l}')